In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
import requests
from rdflib import Graph, URIRef, Literal, Namespace
import io
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [6]:
#VIRTUOSO_URL = "http://localhost:8890/sparql-graph-crud"
VIRTUOSO_URL = "http://localhost:8890/sparql"
GRAPH_URI = "http://example.com/Gent-Terneuzen"
USERNAME = "dba"
PASSWORD = "dba"
AUTH  = (USERNAME,PASSWORD)

In [7]:
params = {'graph': GRAPH_URI}
headers = {'Accept': 'text/turtle'}

# Identifying unique sensors

In [8]:
sensor_set = set()

sensor_query = f"""
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    SELECT  DISTINCT ?sensor
    WHERE {{ 
        GRAPH <{GRAPH_URI}> {{ 
            ?obs a sosa:Observation ;
                 sosa:madeBySensor ?sensor .
        }} 
    }} 
    """
res = requests.get(VIRTUOSO_URL, params={'query': sensor_query, 'format': 'application/sparql-results+json'})
if res.status_code != 200:
        print(f"Error: {res.status_code}")
        print("Response:", res.text)
else:
    print("Unique sensors identified successfully!")

data = res.json()
bindings = data['results']['bindings']

for row in bindings:
    # Extract the URI string and add it to the set
    sensor_uri = row['sensor']['value']
    sensor_set.add(sensor_uri)

print(f"Added {len(sensor_set)} unique sensors to the set.")
print("Sensors:", sensor_set)


Unique sensors identified successfully!
Added 4 unique sensors to the set.
Sensors: {'http://example.com/waterinfo/289441042', 'http://example.com/waterinfo/289435042', 'http://example.com/waterinfo/289429042', 'http://example.com/waterinfo/289423042'}


In [9]:


# 1. Initialize an empty DataFrame for the master table
# We will start with just the 'time' column or an empty DF
final_df = pd.DataFrame()

print("Fetching and pivoting sensor data...")

for sensor_uri in sensor_set:
    # Use the sensor URI (or just the ID part) as the column name
    column_name = sensor_uri.split('/')[-1] 
    
    query = f"""
        PREFIX sosa: <http://www.w3.org/ns/sosa/>
        PREFIX ex: <http://example.com/attributes/>
        SELECT ?time ?value ?unixtime
        WHERE {{ 
            GRAPH <{GRAPH_URI}> {{ 
                ?obs a sosa:Observation ;
                    sosa:resultTime ?time ;
                    sosa:hasSimpleResult ?value ;
                    ex:unixTimestamp ?unixtime ;
                    sosa:madeBySensor <{sensor_uri}> .
            }} 
        }} 
    """
    res = requests.get(VIRTUOSO_URL, params={'query': query, 'format': 'application/sparql-results+json'})
    
    if res.status_code == 200:
        bindings = res.json()['results']['bindings']
        
        # 2. Create a temporary DF for THIS sensor
        temp_data = [
            {'time': row['time']['value'], column_name: float(row['value']['value']), 'unixtime': int(row['unixtime']['value'])} 
            for row in bindings
        ]
        temp_df = pd.DataFrame(temp_data)
        
        if not temp_df.empty:
            temp_df['time'] = pd.to_datetime(temp_df['time'])
            
            # 3. Merge this sensor into the final_df
            if final_df.empty:
                final_df = temp_df
            else:
                # 'outer' join ensures we keep timestamps even if some sensors are missing data
                final_df = pd.merge(final_df, temp_df, on=['time', 'unixtime'], how='outer')

            
            print(f"Added column for sensor: {column_name}")

# 4. Final touches
final_df = final_df.sort_values('time').set_index('time')

print("Finished!")
print(final_df.head())

Fetching and pivoting sensor data...
Added column for sensor: 289441042
Added column for sensor: 289435042
Added column for sensor: 289429042
Added column for sensor: 289423042
Finished!
                           289441042    unixtime  289435042  289429042  \
time                                                                     
2021-03-03 23:15:00+00:00    4797.72  1614813300    3606.54    1605.77   
2021-03-03 23:30:00+00:00    4717.80  1614814200        NaN        NaN   
2021-03-03 23:45:00+00:00    4745.36  1614815100    3556.45    1602.34   
2021-03-04 00:00:00+00:00        NaN  1614816000        NaN    1611.26   
2021-03-04 00:15:00+00:00        NaN  1614816900        NaN    1614.24   

                           289423042  
time                                  
2021-03-03 23:15:00+00:00        NaN  
2021-03-03 23:30:00+00:00        NaN  
2021-03-03 23:45:00+00:00     902.27  
2021-03-04 00:00:00+00:00        NaN  
2021-03-04 00:15:00+00:00     901.96  


In [3]:
X = np.array([[[i/10] for i in range(j, j+5)] for j in range(100)])
y = np.array([(j+5)/10 for j in range(100)])

# 2. Build the Model
model = Sequential([
    # We define 5 time steps and 1 feature (the number itself)
    SimpleRNN(64, input_shape=(5, 1), activation='relu'),
    Dense(1) # The output is the single predicted next number
])

model.compile(optimizer='adam', loss='mse')

# 3. Train
model.fit(X, y, epochs=20, verbose=0)

c:\Users\AYMANS\AppData\Local\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [4]:
# 4. Predict
test_input = np.array([[[0.6], [0.7], [0.8], [0.9], [1.0]]])
prediction = model.predict(test_input)

print(f"Input sequence: [0.6, 0.7, 0.8, 0.9, 1.0]")
print(f"Predicted next value: {prediction[0][0]:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
Input sequence: [0.6, 0.7, 0.8, 0.9, 1.0]
Predicted next value: 0.93
